# Modeling Honolulu Airport Hub Operations Under Reduced Air Traffic Capacity Constraints
### Appendix E

### Authors: Allison Kane and Bernice Ramos
### March 15, 2026

## Read Datafile and Import Packages

In [ ]:
# importing packages

# import pulp
from pulp import LpVariable, LpProblem, LpMaximize, LpStatus, value, LpMinimize

import pandas as pd
import numpy as np
import itertools

# read in CSV file
INFILE = "data/island_flights_cleaned.csv"
df = pd.read_csv( INFILE )

print(df)

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


  roundtrip_destination  demand  flighttime  fuel_burned_hr  fuel_cost_hr  \
0               Kahului      21        1.36            1872       1441.44   
1                  Kona      15        1.59            1872       1441.44   
2                 Lihue      16        1.30            1872       1441.44   
3                  Hilo      12        1.82            1872       1441.44   

   fuel_cost_flight  cabin  pilot  cost_staff_normal  cost_staff_grey  
0         1960.3584      3      2             669.82           861.87  
1         2291.8896      3      2             669.82           861.87  
2         1873.8720      3      2             669.82           861.87  
3         2623.4208      3      2             669.82           861.87  


## Define Parameter and Variable Sets

In [2]:
# routes (defined by non-HNL destinations)
R = df["roundtrip_destination"].tolist()

# planes
P = list(range(1, 21))
    # planes id 1 through 20

# departure times
T = list(range(1, 23))
    # ONLY including valid departure times (none in the blackout zone)

# grey zone departure times
T_grey = [1, 2, 3, 4, 21, 22]
    # this is only needed to calculate cost coefficients

# demand dictionary for each route
F_r = {
    "Kahului" : 21,
    "Kona" : 15, 
    "Lihue" : 16, 
    "Hilo" : 12
}

# departure capacity
X_max = 3


In [3]:
# create decision variable set key
x_keys = list(itertools.product(R, T, P)) # this defines ALL the possible combinations of 
    # route, time depart, and plane ID number in the entire parameter set
    

### Define Cost Coefficients

In [4]:
cost_coeffs = {}

for r, t, p in x_keys:
    if t in T_grey:
        staffing_condition = "cost_staff_grey"
    else:
        staffing_condition = "cost_staff_normal"
    staff_cost = df.loc[df['roundtrip_destination'] == r, staffing_condition].iloc[0]
    fuel_cost = df.loc[df['roundtrip_destination'] == r, 'fuel_cost_flight'].iloc[0]
    total_cost = staff_cost + fuel_cost
    cost_coeffs[(r, t, p)] = total_cost

# check to confirm an output
# cost_coeffs[("Hilo", 3, 1)] # should be confirmed

# cost_coeffs[("Kona", 11, 19)] # should be confirmed


### Define Decision Variables

In [5]:
# create decision variable set
x_vars = {}

for r, t, p in x_keys:
    key = (r, t, p)
    x_vars[key] = LpVariable(f"x_{r}_{t}_{p}", 0, 1, cat = "Binary")
    # this loops through all possible combinations of keyed decision variables, creates
    # a binary LP variable and then stores in x_vars with that key



In [6]:
# confirm outputs of these LP variables
# x_vars[("Kona", 11, 19)] # should be x_Kona_11_19, confirmed
x_vars[("Hilo", 3, 1)] # should be x_Hilo_3_1, confirmed

x_Hilo_3_1

## Define Objective Function

In [7]:
# defines the problem
prob = LpProblem("problem", LpMinimize)

In [8]:
# defines the objective
prob += sum(cost_coeffs[k] * x_vars[k] for k in x_vars)
    # where k represents the key (r, t, p)

## Define Constraints
### Daily Route Flight Minimum Constraint

In [9]:
for r in R:
    prob += sum(x_vars[(r, t, p)] for p in P for t in T) == F_r[r]

### Round-Trip Plane Occupied Constraint

In [10]:
for p in P:
    for t in T[:-1]: # for all takeoff times (2:00 to 21:00, but 22 avoided bc x23 does not exist
        prob += sum(x_vars[(r, t, p)] for r in R) + sum(x_vars[(r, t + 1, p)] for r in R) <= 1

### Air Traffic Control Flight Departures per Hour Constraint

In [11]:
for t in T:
    prob += sum(x_vars[(r, t, p)] for r in R for p in P) <= X_max

## Solve Problem

In [12]:
# solve the problem
status = prob.solve()
print(f"status={LpStatus[status]}")

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/anaconda3/lib/python3.12/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/c0/kjpxq8650bg5t6zt65vr8ymw0000gn/T/6e6f45cfa9de4e5d98a7dbb26a203fe8-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/c0/kjpxq8650bg5t6zt65vr8ymw0000gn/T/6e6f45cfa9de4e5d98a7dbb26a203fe8-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 451 COLUMNS
At line 12612 RHS
At line 13059 BOUNDS
At line 14820 ENDATA
Problem MODEL has 446 rows, 1760 columns and 6880 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 182950 - 0.00 seconds
Cgl0004I processed model has 446 rows, 1760 columns (1760 integer (1760 of which binary)) and 6880 elements
Cbc0038I Initial state - 0 integers unsatisfied sum - 0
Cbc0038I Solution found of 182950
Cbc0038I Before mini branch and bound, 1760 intege

In [13]:
# create a scheduler table
rows = []
for variable in prob.variables():
    if variable.varValue == 1.0:
        ids = variable.name.split("_")
        rows.append(ids[1:])
    else:
        continue

flown_flights = pd.DataFrame(rows, columns = ["Routes", "Departure", "PlaneID"])
flown_flights["Departure"] = pd.to_numeric(flown_flights["Departure"])
flown_flights["PlaneID"] = pd.to_numeric(flown_flights["PlaneID"])

## Confirming Route Minimum Frequency

For model to be valid, the required number of round-trip flights must be scheduled to each destination.

In [14]:
for r in R:
    filtered_df = flown_flights[flown_flights["Routes"] == r]
    print(filtered_df.sort_values(by = "Departure", ascending = True))

     Routes  Departure  PlaneID
28  Kahului          2        5
29  Kahului          3       16
30  Kahului          5        2
31  Kahului          8        9
32  Kahului          9       12
12  Kahului         10       20
13  Kahului         11        8
14  Kahului         12        9
16  Kahului         13        3
15  Kahului         13       17
17  Kahului         14        6
18  Kahului         15        9
20  Kahului         16       18
19  Kahului         16       11
21  Kahului         17        4
22  Kahului         18        6
23  Kahului         18        9
24  Kahului         20       18
25  Kahului         20       19
26  Kahului         21       10
27  Kahului         21        5
   Routes  Departure  PlaneID
42   Kona          3        9
41   Kona          3        8
43   Kona          5        6
44   Kona          6       14
45   Kona          6        5
46   Kona          8        1
47   Kona          9        3
33   Kona         10       17
34   Kona         11      

## Confirming Plane Utilization

For the model to be valid, no plane can be scheduled to fly multiple routes at the same time. No plane can be scheduled to depart in adjacent hours, due to the two-hour roundtrip flight time and turnaround time. 

In [15]:
for p in P:
    filtered_df = flown_flights[flown_flights["PlaneID"] == p]
    print(filtered_df.sort_values(by = "Departure", ascending = True))

   Routes  Departure  PlaneID
46   Kona          8        1
48  Lihue         12        1
51  Lihue         15        1
     Routes  Departure  PlaneID
5      Hilo          1        2
30  Kahului          5        2
2      Hilo         17        2
     Routes  Departure  PlaneID
47     Kona          9        3
16  Kahului         13        3
3      Hilo         18        3
     Routes  Departure  PlaneID
21  Kahului         17        4
40     Kona         20        4
     Routes  Departure  PlaneID
28  Kahului          2        5
45     Kona          6        5
35     Kona         11        5
27  Kahului         21        5
     Routes  Departure  PlaneID
43     Kona          5        6
17  Kahului         14        6
38     Kona         16        6
22  Kahului         18        6
   Routes  Departure  PlaneID
6    Hilo          1        7
57  Lihue          4        7
1    Hilo         14        7
     Routes  Departure  PlaneID
41     Kona          3        8
61    Lihue          7  

## Confirm Flight Traffic Constraint

In [16]:
# flown_flights.sort_values(by = "PlaneID", ascending = True)

for t in T:
    filtered_df = flown_flights[flown_flights["Departure"] == t]
    print(filtered_df.sort_values(by = "PlaneID", ascending = True))

  Routes  Departure  PlaneID
5   Hilo          1        2
6   Hilo          1        7
4   Hilo          1       13
     Routes  Departure  PlaneID
28  Kahului          2        5
10     Hilo          2       14
55    Lihue          2       15
     Routes  Departure  PlaneID
41     Kona          3        8
42     Kona          3        9
29  Kahului          3       16
   Routes  Departure  PlaneID
57  Lihue          4        7
56  Lihue          4       20
     Routes  Departure  PlaneID
30  Kahului          5        2
43     Kona          5        6
58    Lihue          5       15
   Routes  Departure  PlaneID
45   Kona          6        5
44   Kona          6       14
11   Hilo          6       20
   Routes  Departure  PlaneID
61  Lihue          7        8
59  Lihue          7       10
60  Lihue          7       11
     Routes  Departure  PlaneID
46     Kona          8        1
31  Kahului          8        9
62    Lihue          8       19
     Routes  Departure  PlaneID
47     Kon

# Building Final Flight Schedule: HNL Roundtrip Inter-Island Service

In [17]:
schedule = flown_flights.sort_values(by = ["Departure", "PlaneID"], ascending = [True, True])

schedule.reset_index(drop = True)

,Routes,Departure,PlaneID
0,Hilo,1,2
1,Hilo,1,7
2,Hilo,1,13
3,Kahului,2,5
4,Hilo,2,14
...,...,...,...
59,Kahului,21,5
60,Kahului,21,10
61,Hilo,22,12
62,Hilo,22,15


In [19]:
schedule.to_excel("AppendixF_HNL_scheduled_flights.xlsx", index = False)    